In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *


# 0. Create Gold schema

spark.sql("CREATE SCHEMA IF NOT EXISTS hr_catalog_divansh.gold")


# 1. Define schema (IMPORTANT)

salary_schema = StructType([
    StructField("salary_id", StringType(), True),
    StructField("employee_id", StringType(), True),
    StructField("base_salary", DoubleType(), True),
    StructField("bonus_pct", DoubleType(), True),
    StructField("effective_date", StringType(), True),
    StructField("currency", StringType(), True)
])


# 2. Auto Loader source

salary_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/FileStore/hr_divansh/schema/salary_stream")
    .option("header", True)
    .option("pathGlobFilter", "salaries*.csv")   # ✅ ONLY salary files
    .schema(salary_schema)
    .load("/FileStore/hr_divansh/raw/")
)


# 3. Transform + watermark

salary_stream = (salary_stream
    # Safe timestamp conversion
    .withColumn(
        "effective_date",
        F.expr("try_to_timestamp(effective_date, 'yyyy-MM-dd')")
    )
    # Drop bad records (optional but recommended)
    .filter(F.col("effective_date").isNotNull())
    # Watermark for late data
    .withWatermark("effective_date", "1 day")
    # Idempotency (no duplicates)
    .dropDuplicates(["salary_id"])
)


# 4. Write stream to Delta

query = (salary_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/FileStore/hr_divansh/checkpoints/live_salary")
    .outputMode("append")
    .trigger(availableNow=True)   # best for assignment
    .toTable("hr_catalog_divansh.gold.live_salary_updates")
)


# 5. Verify duplicates (idempotency check)

spark.sql("""
SELECT salary_id, COUNT(*) AS cnt
FROM hr_catalog_divansh.gold.live_salary_updates
GROUP BY salary_id
HAVING cnt > 1
""").show()